# Week 3 review · 23–29 Nov 2026 (London time)

**Repository:** `affordance-vla` · written on 26 Nov 2026 (the week is not over)

**Order:** 0 summary · 1 Pipeline v4 · 2 why RoboCasa365 · 3 step 0 ①: mug transfer · 4 step 0 ②: RoboCasa365 on the server · 5 HOT3D · 6 affordance-extraction methods · 7 try it · 8 timeline and next week

- Figures first, then a few lines. Every number is copied from a task record (named in each section); the notebook computes nothing.
- **Simulator ground truth** (object poses, meshes) is used only to make labels and to evaluate, never as a model input; each section says which of its numbers are real runs and which come from the literature.
- **[designed]** = in the plan, not built; **[done]** = code ran and has results.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "affordance_vla").is_dir())

## 0 · Summary

**Scope changed (24–25 Nov).** The advisor asked for a VLA that is not limited to one kind of task: it should extend to tasks in indoor rooms and show some video understanding. The project lead picked option 1B (RoboCasa365 kitchens as the main environment, LIBERO kept to connect to earlier results) and said compute is not a constraint for now. **Pipeline v4** (approved 25 Nov) adds an understanding part that names the next object to touch, to the v3 core (interaction map, obedient action).

**What was done this week**

| Section | Work | Kind of evidence |
| --- | --- | --- |
| 1 | Pipeline v4 figure, v3 to v4 changes | design; nothing built for v4 yet |
| 2 | dataset survey, why RoboCasa365 | literature (abstracts, project pages, data cards); nothing downloaded |
| 3 | step 0 ①: carry the mug map to a new mug (AFFO-9) | computed from replayed LIBERO demonstrations; the object poses and meshes are simulator ground truth, the NDF weights are public |
| 4 | step 0 ②: RoboCasa365 on the server (AFFO-10) | real simulator runs (software rendering) and real starVLA training and evaluation jobs; no ground truth fed to a model |
| 5 | HOT3D: can human-hand data give our map (AFFO-11) | real motion-capture annotations (ground truth by construction), no model |
| 6 | survey of affordance-extraction methods | literature; nothing run |

**What we found**

- **Step 0 ①: 1 of 6 groups passes** (red mug held out, NDF). The main reason is in the demonstration data (each mug is always placed in one orientation, so demonstrators grasp from one side), not in the transfer code (inference). With a fixed orientation, "unseen shape" and "unseen orientation" cannot be separated on these mugs.
- **Step 0 ②: the pipeline works end to end.** RoboCasa365 renders with no display or GPU, demonstrations replay from saved states with depth and segmentation added, starVLA trains and evaluates. The **0/30 successes is not a capability result**: a model trained for 300 steps, evaluated with one camera where training used three; it only shows that the pipeline runs, at about 4.8 minutes per episode.
- **A single 48 GB card is too small for QwenPI**, even with the VLM frozen (action head alone 3.35 B parameters). QwenOFT with a frozen VLM needs 11.4 GiB; QwenOFT full fine-tuning did not fit either. **Which backbone and training mode step 1 uses is an open decision for the project lead**, not settled here.
- **HOT3D can fill our map format** from hand and object meshes, but gives "how a person holds it", not "how a gripper should grasp it"; it will not enter the main line. The result is a list of what the data-collection team should record (section 5).
- **Two survey numbers were corrected by measurement** (section 2): RoboCasa365 demonstrations are LeRobot v2.1, and the 50 target bundles are 61.6 GB.

**Where we are, and what is not decided.** Step 0 is finished. **Step 1 has not been approved**, and the plan for next week is not settled: it waits on two questions to the project lead (backbone and training mode; when step 1 starts and how many GPUs it may use). Section 8 lists the candidate next steps, as candidates.

**Decisions this week**

| Date | Decision | By |
| --- | --- | --- |
| 24 Nov | survey data sets and extraction methods first, then pick the final experiment scope | project lead |
| 24 Nov | agents run on the Mac and control the server; one branch and worktree per issue, direct push of `agent/*` branches; GPU 0 first | project lead |
| 25 Nov | scope 1B; compute not a constraint for now | project lead |
| 25 Nov | Pipeline v4 approved; step 0 starts, one GPU through the queue | project lead |
| 25 Nov | this weekly review is the one document the project lead reads | project lead |

## 1 · Pipeline v4

Approved by the project lead on 25 Nov. One sentence: **understanding names the next object to touch, the interaction map says where to touch it and in which direction, and the action is trained to obey that plan.** The thesis is unchanged (affordance helps only if the action is trained to obey it); v4 adds who decides which object the plan points at. Source: `docs/vla/2026-11-25_VLA_branch_Pipeline_v4.md`.

In [ ]:
Image(filename=str(ROOT / "report" / "media" / "pipeline_overview" / "v4_model.png"), width=1100)

*Gold = open-source starVLA; orange = our parts built and trained for LIBERO in v3; grey = ours, no weights; purple dashed = new in v4, **designed, not built**. The frame is a recorded RoboCasa365 demonstration. Nothing in this figure runs on RoboCasa365 yet: the orange parts exist for LIBERO, and the backbone (QwenPI or QwenOFT) is an open decision (section 0).*

**What changed from v3 to v4** (Pipeline v4, section 2; items marked *design* have no experiment behind them):

| | v3 | v4 |
| --- | --- | --- |
| Environment | LIBERO (tabletop), zero-shot on LIBERO-PRO | RoboCasa365 kitchens; LIBERO and LIBERO-PRO kept to connect to earlier results |
| Tasks | pick and place | atomic tasks (pick and place, open drawer, open door, knob), then composite tasks of several steps |
| Interaction map | grasp contact and closing direction | also drawer and door handles and knobs, plus a direction to move after touching *(design)* |
| Understanding | none; the semantic gate picks an object from one frame and the instruction | a new part: reads the recent frames and the instruction, answers which step we are in and what the next interaction target is |
| Who picks the target | the instruction | the understanding part *(design)* |
| Unseen instances of a category | 3 mugs, plus scaled mugs | instances RoboCasa holds out: 27 mugs, 22 bowls, 22 cups |
| Carrying a map to a new instance | nearest point | nearest point and NDF, compared (step 0 ①, section 3) |
| Training scale | frozen-VLM diagnostics, 1 training seed | main results with full fine-tuning, at least 3 training seeds *(plan)*; light training only for quick comparisons |
| Compute | planned for one 8×A6000 server | not a constraint for now (project lead, 25 Nov); full fine-tuning on one 48 GB card is still unsolved (section 4) |

Unchanged: starVLA as backbone family, the model works on 2D images (map plus depth), simulator ground truth is used for labels and evaluation only, never as a model input at test time.

## 2 · Why RoboCasa365

The project lead's advisor asked (24 Nov) for a VLA that is not limited to one kind of task: it should extend to tasks in indoor rooms and show some video understanding. We needed one simulator with several kinds of interaction, room-level scenes and demonstrations with fine actions. **RoboCasa365** (2026) is that: kitchens, with demonstrations. Three of its tasks as the recorded demonstrations show them (a pick-and-place, a drawer, a knob); three cameras, 256 × 256, 20 fps:

In [ ]:
Image(filename=str(ROOT / "report" / "media" / "weekly_2026_11_23" / "robocasa365" / "three_cameras.png"), width=900)

**What the survey compared** (AFFO-4, 24 Nov; abstracts, project pages and data cards only, **nothing was downloaded or run**; "not found" = the survey did not find it, not that it does not exist). Compressed from 11 candidates to the 8 that matter:

| Dataset | Scene | What it gives | Scale | Closed-loop test in simulation | Main limit |
| --- | --- | --- | --- | --- | --- |
| **RoboCasa365** | room-level kitchens, mobile manipulation | 3 RGB cameras, language, robot actions; layout and object files per episode | 365 tasks, 2500+ kitchens, 2200+ h | yes; 50-task leaderboard (best VLA 62.6) | kitchens only; no depth or segmentation, no question-answer labels |
| BEHAVIOR-1K (2025 challenge) | whole houses, whole-body mobile | RGB-D, instance segmentation, object states, skill labels | 10,000 episodes, 1200+ h, ~1.5 TB | yes; best q-score 0.26 | object meshes are encrypted (no category-level labels); hard, 8×H200 for a month for the winner |
| MS-HAB | apartments (ReplicaCAD) | RGB-D 128², state | 84 apartments | yes; RL/IL baselines only | no VLA numbers; license not found |
| LIBERO / PRO | tabletop | wrist + third-person cameras | 50 demos × 130 tasks | yes (our current setting) | 2–8 movable objects per scene |
| DROID | 564 real scenes, fixed arm | 3 cameras, language | 76k episodes, 350 h, 1.7 TB (RLDS) | only a separate IsaacLab kitchen set (π0.5: 38.3%) | RLDS has no depth; sim scenes are not the data's scenes |
| AgiBot World | 106 real scenes, mobile dual arm | 8 cameras, sub-step labels | 1M episodes, 2976 h, 48 TB | not found | non-commercial; huge |
| HD-EPIC | 9 real kitchens, first person | digital twin of each kitchen, 3D masks, 26.6K questions | 41 h, 2.3 TB (about 120 GB for video + labels) | none (a question benchmark; Gemini Pro 37.6%) | no robot actions, no grasp labels; non-commercial |
| EgoDex | tabletop, first person | 3D hand poses (68 joints) | 829 h | none | no scene 3D; non-commercial |

(Also in the survey: EPIC-KITCHENS + VISOR, HOI4D, SceneFun3D; none has robot actions.)

- **No dataset has both understanding labels and fine actions** (survey conclusion 2). BEHAVIOR-1K comes closest but is the hardest to use.
- **Why RoboCasa365 (survey, then step 0 ②):** the only candidate with room-level scenes, fine actions, several interaction types, a leaderboard with VLAs and plain MuJoCo. Understanding labels have to be generated from simulator state and sub-task labels (design, not built).
- **Two survey numbers were wrong or unchecked, measured by AFFO-10 (section 4):** the demonstrations are LeRobot v2.1 (the survey said v3.0), and the 50 target bundles are 61.6 GB (the survey said 188 GB; the source of that figure was not found).
- **Not adopted:** BEHAVIOR-1K (encrypted meshes, compute); HD-EPIC is kept for later as an understanding source (Pipeline v4, section 8).

## 3 · Step 0 ①: carrying the mug map to a mug nobody demonstrated (AFFO-9)

The check: build the map from two mugs' demonstrations, carry it to the third mug, and compare it with where the demonstrators grasped that third mug. Each figure: *left* nearest-point transfer in the category frame, *middle* NDF transfer, *right* the mug's own demonstrations; orange = the held-out mug's own grasps, projected onto one demonstration frame.

In [ ]:
MUG_FIGURES = ROOT / "report" / "media" / "weekly_2026_11_23" / "mug_transfer"
for name in ('porcelain_mug', 'white_yellow_mug', 'red_coffee_mug'):
    display(Image(filename=str(MUG_FIGURES / f"{name}.png"), width=1000))

| Held-out mug (own grasps) | Method (source grasps) | Landing rate (pass ≥ 0.8) | Median axis error (pass < 20°) |
| --- | --- | --- | --- |
| white `porcelain_mug` (269) | nearest point (607) | 0.00 ✗ | 62.7° ✗ |
| | NDF (200) | 0.00 ✗ | 45.1° ✗ |
| yellow-white `white_yellow_mug` (334) | nearest point (542) | 0.02 ✗ | 15.2° ✓ |
| | NDF (200) | 0.70 ✗ | 18.6° ✓ |
| red `red_coffee_mug` (273) | nearest point (603) | 0.00 ✗ | 9.0° ✓ |
| | NDF (200) | 1.00 ✓ | 9.6° ✓ |

- **1 of 6 pairs passes** (red mug held out, NDF); nearest point passes 0 of 3, NDF 1 of 3. NDF's landing rate is never below nearest point's.
- **The map itself is stable:** a mug's own demonstrations split in half agree at 0.97–1.00.
- **The data, not the transfer code, is the main limit (inference):** LIBERO fixes each mug's start orientation, so demonstrators always grasp from one side. The white mug's handle points the other way from the other two; its grasps lie ~30° from the handle in the category frame, the others' ~170°. Turning the category frame to the scene instead gives 1.00 / 0.77 / 1.00.
- **NDF was one seed**, 100 sampled grasps per source mug, no tuning: a single red-mug win is not a general result.
- **Consequence for the plan:** with fixed orientations, "unseen shape" and "unseen orientation" cannot be separated on these mugs.

```bash
# repository root of the A6000 server; W=/mnt/nas/xinyizheng/affordance_vla
bash scripts/data/run_mug_transfer_check.sh collect                       # CPU
bash scripts/server/submit_queue_job.sh 164 mug_ndf 8000 1 "cards=<free cards>" -- \
    bash $PWD/scripts/data/run_mug_transfer_check.sh ndf                  # GPU through the queue, ~45 min
bash scripts/data/run_mug_transfer_check.sh evaluate                      # CPU: table and figures
cat $W/runs/mug_transfer/mug_mesh/results.md
```

## 4 · Step 0 ②: RoboCasa365 runs on the server (AFFO-10)

Three of the 50 target tasks (**CoffeeSetupMug**, **OpenDrawer**, **TurnOffStove**; frames in section 2) were used for every check below.

**Replay.** From the saved MuJoCo model and state of each step, the simulator renders the same view again (software
rendering, no GPU) and adds what the videos do not hold: depth and instance segmentation.

In [ ]:
Image(filename=str(ROOT / "report/media/weekly_2026_11_23/robocasa365/replay_video_vs_rerender.png"), width=1000)

**Object meshes.** The category-level map needs surface points of every instance of a category. RoboCasa365 ships
27 mugs, 22 cups and 22 bowls; its own split holds out 10, 7 and 7 of them for the target kitchens.

In [ ]:
Image(filename=str(ROOT / "report/media/weekly_2026_11_23/robocasa365/category_surface_points.png"), width=1000)

### What we found

- **It installs and renders without a display or a GPU:** software rendering, 0.10–0.11 s per 256 × 256 camera view, 0.054–0.061 s per physics step; rebuilding one demonstration's kitchen takes 16–19 s.
- **Replay works:** 15 demonstrations (3 tasks × 5) re-rendered from their saved states match the recorded video at 34–36 dB PSNR (one frame off: 26–29 dB) and now come with depth and instance segmentation. Replaying the *actions* open loop does not reproduce the recorded states bit for bit (state error 0.05–0.8 typical, up to 5.9).
- **The data differs from the survey:** LeRobot v2.1 (not v3.0), 514 demonstrations per task, 61.6 GB for the 50 target bundles (not 188 GB; source of that figure not found).
- **Held-out objects are built in:** `split="target"` holds out the last ≤ 5 instances of every category and uses 10 fixed kitchens; the leaderboard protocol and the demonstrations use it. Mugs: 27 instances, 10 held out; cups 22 / 7; bowls 22 / 7; all export surface points.
- **Composite demonstrations carry per-frame subtask labels** (language, skill name, stage, index), as documented; atomic ones do not.
- **No public RoboCasa365 weights** (the public RoboCasa checkpoints are the GR1 tabletop benchmark).
- **QwenOFT trains and evaluates:** frozen VLM, batch 8, 300 steps: 11.4 GiB, 0.445 s/step. Evaluation of that 300-step model, 10 episodes on each of the 3 tasks, 3 clients in parallel: 4.2–5.2 min per episode (4.8 on average, about 0.41 s per environment step), 0/30 successes (a speed check of an under-trained model with a camera and state mismatch, not a capability).
- **QwenPI does not fit one 48 GB card:** its action head alone has 3.35 B parameters (7.79 B in total); out of memory with a frozen VLM at batch 8 and 1, and with all weights at batch 1. Full fine-tuning of QwenOFT (4.5 B) at batch 1 does not fit either (needs roughly 70 GiB, inference). Step 1 needs a frozen QwenOFT VLM, LoRA on frozen weights, or several cards.

### Try it (server `estar`, repository root of the task branch; all CPU except the last two, which go through the queue)

```bash
bash envs/server/setup_robocasa365_env.sh                    # conda env + pinned code + 22 GB of assets on the NAS
bash scripts/data/fetch_robocasa365.sh                       # 3 tasks, 1.4 GB (--dry-run prints sizes only)
source envs/server/robocasa365_env.sh
"$RC365_PYTHON" scripts/data/replay_robocasa365.py --task CoffeeSetupMug OpenDrawer TurnOffStove --episodes 5 \
    --out $W/runs/robocasa365_replay/my_run                 # re-render, depth, segmentation, PSNR against the videos
"$RC365_PYTHON" scripts/report/make_rc365_figures.py --replay $W/runs/robocasa365_replay/my_run --out /tmp/rc365_figures
bash scripts/server/submit_queue_job.sh 164 rc365_train 30000 1 "cards=6 7" -- \
    env RUN=my_train CONFIG=$PWD/configs/rc365/qwenoft_rc365_3tasks.yaml bash $PWD/scripts/rc365/run_train_gpu.sh
bash scripts/server/submit_queue_job.sh 164 rc365_eval 20000 1 "cards=6 7" -- \
    env CKPT=$W/runs/robocasa365_starvla/my_train/checkpoints/steps_300_pytorch_model.pt \
    bash $PWD/scripts/rc365/run_eval_gpu.sh
```

`cards=6 7` is what was free when AFFO-10 ran (GPU 0 was full then). **Use `cards=0` first; if GPU 0 is full, list the cards that have room at that moment.** The two queue commands need a GPU and were run by AFFO-10, not re-run for this review.

RoboCasa365 was only run on the server (22 GB of assets, 1.4 GB of demonstrations for these three tasks); no Mac command is given because none was tried.

## 5 · HOT3D: can human-hand data give our interaction map?

**Real data, ground-truth annotations** (motion capture), no model involved: [HOT3D](https://facebookresearch.github.io/hot3d/) (Meta, CVPR 2025) = egocentric recordings of 19 people handling 33 rigid objects, with per-frame 6-DoF object poses, hand meshes and scanned object meshes. Below: two 5-second clips of one person holding the white mug (`P0015_dbe00981`).

In [ ]:
from pathlib import Path
from IPython.display import Image
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "affordance_vla").is_dir())
HOT3D = ROOT / "report" / "media" / "weekly_2026_11_23" / "hot3d"
Image(filename=str(HOT3D / "overlay.png"), width=620)

*Top of the figure above: the annotated hand meshes (thumb in orange) and the mug mesh drawn on the camera frame (Aria RGB, fisheye). Below: the same 10 seconds turned into labels.*

In [ ]:
Image(filename=str(HOT3D / "contact.png"), width=1000)

*Top row: how often each point of the mug is within 5 mm of a hand point (300 frames; grey = never). Cyan lines: the thumb-to-fingers closing axes of the 5 held grasps found. Bottom row: the same grasps turned into a map of `interaction_labels.py`, the format of our demonstration maps.*

- **Yes, it fills our map format.** Surface points (scanned mesh), grasps in the object frame (hand mesh + object pose) and the closing axis (thumb pad → finger pads) all come out; the result loads with `load_maps`.
- **Three things are missing:** (1) a label for "this is a held grasp" (we guess it from thumb–finger opposition, ≥ 5 frames); (2) the step from a human hand to a parallel gripper (the 9 cm mug body is at the edge of our `MAX_REACH_M`); (3) enough shapes per category: 2 mugs, 1 bowl, 0 cups, rigid objects only (no drawers, doors, knobs).
- **Numbers:** 145 of 300 frames have a hand within 5 mm of the mug (143 / 147 at 3 / 10 mm); 34% of its surface is touched at 5 mm (20% / 65%); 5 held grasps, 75–93 mm between the pads; the closing axis of one continuous hold drifts by 48° and 71° (the hand re-grips), so a hold is split where it turns 20°.
- **Not in HOT3D:** contact, action, task or language labels; depth; any robot. Hand annotations are non-commercial (CC BY-NC-SA); the UmeTrack hand mesh needs no MANO licence.
- **For the data-collection team (top 3 of 10 in the record):** per-frame object pose + mesh, per-frame hand mesh in the same frame, and quality masks; then hold start/end, ≥ 8–10 shapes per category (mug, bowl, cup, bottle, can), and articulated handles and knobs.

**Run it** (server, CPU, ~30 s; needs the `hot3d` env from `envs/server/setup_hot3d_env.sh`; not tried on the Mac):

```bash
bash scripts/data/fetch_hot3d_demo.sh          # 2 clips + 3 meshes, 0.6 GB from Hugging Face, no sign-up
source envs/server/hot3d_env.sh
PYTHONPATH=. python scripts/data/hot3d_contact_demo.py --out $W/runs/hot3d_demo/mug_white_v2   # add --threshold-mm 10 to see the threshold matter
```

## 6 · How to extract an affordance for a whole category (survey)

**Question (AFFO-5, 24 Nov):** the way one object category is grasped must carry over to new instances of it. From which data, in which representation, by which transfer? Which one or two routes fit our pipeline? Abstracts and project pages for 10 methods (round 1), then full texts for four of them plus RAM (round 2); **nothing was downloaded or run**. Our own approach (demonstration statistics → a 3D map per category) is not counted among them.

| Method | Data it needs | Output | Transfer to a new instance | Direction? | Cross-instance evidence (as reported) | How it connects | Limit for us |
| --- | --- | --- | --- | --- | --- | --- | --- |
| **NDF** 2021 (2112.05124), full text | point cloud; 5–10 demonstrations | 6-DoF gripper pose | descriptor matching + optimisation | **yes, 3D** | unseen ShapeNet instances: mug upright 0.88 total success, any pose 0.58; 1 demonstration: 0.46; real robot qualitative only | external planner; MIT code | shapes only from ShapeNet; single-view point clouds not tested; no precedent as an offline label maker |
| Robo-ABC 2024 (2401.07487), full text | RGB; human-video memory of 51 categories | 2D contact point | retrieval + diffusion-feature correspondence | no (grasp direction from AnyGrasp) | contact success 60.7% (second best 29.1%); mug not listed alone | external | 2D point, no closing axis |
| RAM 2024 (2407.04689), full text | RGB-D; retrieval memory | 3D contact point + direction after contact | retrieval + feature correspondence | after contact, not gripper pose | simulation 52.6% on 70+ objects, 10 classes | external; also generates demonstrations for distillation | mug only in real-robot videos, no number |
| VRB 2023 (2304.08488), full text | RGB; egocentric human video | 2D contact heat-map + wrist path | end-to-end | 2D path only | cabinet / door / drawer kNN success 0.6 / 1.0 / 1.0; knob not tested; no mug or bowl grasp number | **trains** imitation or RL policies (not a VLA) | 2D, no grasp direction |
| Actron3D 2025 (2510.12971) | a few human RGB demonstrations | object-centric geometry, contact, motion functions | retrieve and align to the target scene | not stated | not found | external; needs RGB-D plus grasp and control modules | extra modules needed |
| Dense Object Nets 2018 | multi-view RGB-D, self-supervised | dense descriptors | nearest descriptor | no | said to generalise to unseen objects; numbers not found | external | points only |
| RoboPoint 2024 | synthetic data | image key-points from language | VLM | no | +21.8% spatial-affordance accuracy over GPT-4o and PIVOT (abstract) | external | no direction |
| Where2Act 2021 | simulated interaction | per-pixel actionability + trajectory | end-to-end | trajectory | cross-category, number not found | external | articulated parts, not grasping a mug |
| kPAM 2019, GG-CNN 2018 | key-point labels / grasp data | 3D key-points / per-pixel grasp | category key-points / end-to-end | no | not found / real 83–88% | external | key-points need hand definition; GG-CNN is only a model-head reference |
| AffordanceVLA 2026 (2606.06155), full text | 8.8M samples | object latent, 2D contact heat-map, 3D shape | end-to-end | no approach direction | LIBERO 95.8 vs 94.4 | **trains** a VLA; weights and data not released | the nearest neighbour; also the one to differ from |

- **Only AffordanceVLA trains a VLA; VRB trains imitation or RL policies; the other eight are external labels or planners.** So the contribution "a training method" has to come from our own obedience training; these methods only produce labels (inference).
- **Recommended route 1: our own category-level 3D map + nearest-point transfer.** It inherits the labels we have, needs no new data, the transfer is geometric and testable. Risk: LIBERO has meshes for only 3 mugs.
- **Recommended route 2: replace the correspondence with a descriptor, NDF first, Robo-ABC or RAM second.** NDF is the only one that carries a 6-DoF relative pose to a new shape. Risk: all are external-planner methods; they supply labels or a transfer step at best.
- **Minimal test the survey proposed (run in step 0 ①, section 3):** leave one mug out, nearest point vs NDF. Result: 1 of 6 pairs passes.
- **Not worth doing now:** VRB (2D, no grasp direction); features-only methods (points without axes) need a second layer to get the closing axis.
- **Not settled:** NDF on single-view point clouds; whether our noisy teleoperated grasps are cleaner or worse than the clean poses NDF expects (inference).

## 7 · Try it

Two groups. **Server**: `ssh estar`, repository root of a worktree of branch `agent/vla-engineer/affo-12` (`git worktree add --detach ~/wt-review origin/agent/vla-engineer/affo-12`), `W=/mnt/nas/xinyizheng/affordance_vla`. **Mac**: repository root of a clone that has run the setup once. "Run" = I ran it for this review on 26 Nov; "not re-run" says why. Output folders are new, so earlier results are never overwritten.

**GPU commands:** the examples written by AFFO-10 use `cards=6 7`, because GPU 0 was full when they ran. **Use GPU 0 first (`cards=0`); if it is full, list the cards that have room at that moment.** GPU jobs go through the queue only (`scripts/server/submit_queue_job.sh`).

### On the server

**① Mug transfer check (section 3).** CPU stages are cheap; the NDF stage needs a GPU.

```bash
bash scripts/data/run_mug_transfer_check.sh collect --set out=runs/mug_transfer/my_check       # 51 s
bash scripts/server/submit_queue_job.sh 164 mug_ndf 8000 1 "cards=0" -- \
    bash $PWD/scripts/data/run_mug_transfer_check.sh ndf --set out=runs/mug_transfer/my_check   # GPU, about 45 min; run by AFFO-9, not re-run
bash scripts/data/run_mug_transfer_check.sh evaluate --set out=runs/mug_transfer/my_check       # table + figures
cat $W/runs/mug_transfer/my_check/results.md
```

Run: `collect` and `evaluate` (with AFFO-9's NDF files copied into the new folder): `results.json` identical to AFFO-9's. The `ndf` stage was not re-run (GPU).

**② HOT3D demo (section 5).** CPU, 30 s.

```bash
bash envs/server/setup_hot3d_env.sh       # once
bash scripts/data/fetch_hot3d_demo.sh     # 2 clips + 3 meshes, 0.6 GB, no sign-up
source envs/server/hot3d_env.sh
PYTHONPATH=. python scripts/data/hot3d_contact_demo.py --out $W/runs/hot3d_demo/my_check   # try --threshold-mm 10
```

Run: the fetch (data already there, nothing downloaded) and the demo (29 s): `summary.json` identical to AFFO-11's. The environment set-up script was not re-run.

**③ RoboCasa365 (sections 2 and 4).** Everything except the last two commands is CPU.

```bash
bash envs/server/setup_robocasa365_env.sh                              # conda env + pinned code + 22 GB assets, safe to repeat
bash scripts/data/fetch_robocasa365.sh --dry-run                       # sizes only; without --dry-run: 1.4 GB for 3 tasks
source envs/server/robocasa365_env.sh
"$RC365_PYTHON" scripts/data/replay_robocasa365.py --task CoffeeSetupMug OpenDrawer TurnOffStove --episodes 5 \
    --out $W/runs/robocasa365_replay/my_run                           # depth, segmentation, PSNR against the recorded video
"$RC365_PYTHON" scripts/report/make_rc365_figures.py --replay $W/runs/robocasa365_replay/my_run --out /tmp/rc365_figures
bash scripts/server/submit_queue_job.sh 164 rc365_train 30000 1 "cards=0" -- \
    env RUN=my_train CONFIG=$PWD/configs/rc365/qwenoft_rc365_3tasks.yaml bash $PWD/scripts/rc365/run_train_gpu.sh   # GPU
bash scripts/server/submit_queue_job.sh 164 rc365_eval 20000 1 "cards=0" -- \
    env CKPT=$W/runs/robocasa365_starvla/my_train/checkpoints/steps_300_pytorch_model.pt \
    bash $PWD/scripts/rc365/run_eval_gpu.sh                            # GPU, about 5 min per episode
```

Run: set-up (11 s, already installed), the dry run (three bundles of 423–479 MB), the replay with `--episodes 1` instead of 5 (3 min 10 s, PSNR 32–39 dB at the recorded frame; section 4 reports 34–36 dB over 15 demonstrations) and the figure script (three PNGs). The 5-episode replay, training and evaluation were run by AFFO-10 and not re-run (GPU, or 15 min).

### On the Mac

**Once:** `bash envs/local/setup_sim_env.sh` (conda env `affordance-vla-sim`), `bash scripts/local/fetch_sim_data.sh` (LIBERO-PRO, task files, evaluation logs, the interaction maps). Neither was re-run: the environment and data were already there. The commands below open the MuJoCo viewer (rotate with the mouse, space pauses, close the window to end) and describe each step in the terminal. A replay re-runs the actions of a server log in the same scene; no model runs on the Mac.

```bash
bash scripts/local/sim.sh list runs/libero_pro/predicted_g1_hold_s10                                    # every logged episode with its outcome
bash scripts/local/sim.sh replay runs/libero_pro/predicted_g1_hold_s10/libero_object_swap/task2_ep0.npz  # position perturbation, ours: succeeds
bash scripts/local/sim.sh replay runs/libero_pro/released/libero_object_swap/task2_ep0.npz               # same scene, released checkpoint: fails
bash scripts/local/sim.sh map --suite libero_10 --task 4                                                # the 3D interaction map of the mugs on the objects
bash scripts/local/sim.sh scene --suite libero_object_swap --task 2 --episode 0                         # the test scene with each object's demonstrated grasp
```

Run: `list`, and the first `replay` with `--set viewer.realtime=0 --set viewer.hold=false` (so it ends by itself): success in 128 steps, the same as the server's log, end-effector drift 0.0 mm. **Not re-run this week:** the second replay, `map` and `scene` (they only open a window and wait for you; they were checked for the week-2 review). More demo commands of week 2 are in section 6 of `report/2026-11-16_weekly_review.ipynb`. **There is no Mac command for sections 3 to 5**: they need the server's data (mug maps and NDF), the HOT3D clips or RoboCasa365 assets; their figures are in this notebook.

## 8 · Timeline and next week

**This week (23–29 Nov, London time).** Source: `docs/HANDOVER.md`, section 3, and the task records.

| Day | What happened |
| --- | --- |
| Mon 23 Nov | week-2 review (English, short) for the group meeting; MuJoCo demo commands; pipeline figures redrawn |
| Tue 24 Nov | the advisor's feedback sets a wider scope; the project moves to Multica; two surveys (data sets, extraction methods) done and accepted; hand-over summary written |
| Wed 25 Nov | scope 1B chosen, **Pipeline v4 approved**, step 0 started (AFFO-9, AFFO-10); HOT3D assessment (AFFO-11); step 0 ① ends with 1 of 6 groups passing; step 0 ② passes; this review started |
| Thu 26 Nov | this review built and executed; the commands of section 7 run |
| Fri 27 – Sun 29 Nov | not yet happened |

The account's usage limit cut off agent runs on 24 and 25 Nov (including the first attempt at this review); each was restarted by the lead.

**Next week: candidates, nothing decided.** Step 1 (6–8 atomic tasks: labels, map head and gate, obedience training) is **not approved**. It waits for the project lead's answers on the backbone (question 7: QwenOFT with a LoRA on one card, QwenPI later with more compute, or QwenOFT with full fine-tuning on several cards) and on the start of step 1 with its GPU limit (question 8). If step 1 starts, things the records say must be handled first:

- **Labels:** the map in object coordinates mixes in "which side the robot came from" (demonstrations cover only the side facing the robot). Two candidates: complete a symmetric part into a ring and pick by robot position, or record grasps in a robot-facing frame; decide on RoboCasa, where mugs are placed at random.
- **starVLA evaluation bridge:** training reads 3 cameras, evaluation sends 1 and builds the state unconditionally, and the evaluation environment has no seed. Align before any success rate is reported.
- **Evaluation budget:** every episode takes about 4.8 minutes; a rough estimate (inference) of 27 hours of simulation for 8 tasks × 2 instance splits × 20 episodes, spread over at most 30 CPU processes.
- **Unseen instances:** the target-split demonstrations were recorded with the held-out instances, so "seen and unseen instances" must be designed by us (idea, not checked: train on the pretrain split, evaluate on the target split).
- **One known test problem:** `tests/test_config.py::test_the_resolved_config_records_its_source` fails when pushed from a task worktree on the Mac (the hook exports `GIT_DIR`); it passes when run directly.